In [ ]:
import sys, types

fake_audio = types.ModuleType('mediapipe.tasks.python.audio')
for sub in ['audio_classifier', 'audio_embedder', 'core']:
    fake_sub = types.ModuleType(sub)
    setattr(fake_audio, sub, fake_sub)
    sys.modules[f'mediapipe.tasks.python.audio.{sub}'] = fake_sub
sys.modules['mediapipe.tasks.python.audio'] = fake_audio

print("Monkey-patch applied")

Monkey-patch applied


In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('mediapipe', 'protobuf==5.29.4', 'opencv-python-headless',
    'xgboost', 'shap', 'scipy', 'pyyaml', 'openpyxl',
    'torch', 'torchvision', 'scikit-learn', 'ipywidgets')

import mediapipe as mp
print('mediapipe:', mp.__version__)
print('Install complete — restart runtime/kernel before running Cell 2')

mediapipe: 0.10.35
Install complete — restart runtime/kernel before running Cell 2


In [ ]:
import sys, types

fake_audio = types.ModuleType('mediapipe.tasks.python.audio')
for sub in ['audio_classifier', 'audio_embedder', 'core']:
    fake_sub = types.ModuleType(sub)
    setattr(fake_audio, sub, fake_sub)
    sys.modules[f'mediapipe.tasks.python.audio.{sub}'] = fake_sub
sys.modules['mediapipe.tasks.python.audio'] = fake_audio

# verify it works now
import mediapipe as mp
from mediapipe.tasks import python as mptasks
from mediapipe.tasks.python import vision as mpvision
print("OK:", mp.__version__)

OK: 0.10.35


In [ ]:
import os, json, time, copy, pickle, urllib.request, logging
from datetime import datetime
from typing import List, Optional
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.signal import savgol_filter, find_peaks
from scipy.interpolate import interp1d
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import pearsonr

# ── Environment detection ────────────────────────────────────────────
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running in Colab — Drive mounted')
except ImportError:
    print('Running in VS Code / local Jupyter')

# ── Device ───────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Data paths ───────────────────────────────────────────────────────
# Edit these to match where your CSV files actually are.
if IN_COLAB:
    DATA_DIR   = '/content/drive/MyDrive/datasets'
    MODEL_SAVE = '/content/drive/MyDrive/models/rehabnet_best.pth'
else:
    DATA_DIR   = './datasets'          # local folder next to this file
    MODEL_SAVE = './models/rehabnet_best.pth'

UIPRMD_CSV = os.path.join(DATA_DIR, 'uiprmd.csv')
KIMORE_CSV = os.path.join(DATA_DIR, 'squats_tabular_timeseries_binary.csv')
KERAAL_CSV = os.path.join(DATA_DIR, 'keraal_final_binary.csv')

os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)

# ── Sequence constants ────────────────────────────────────────────────
TARGET_LEN  = 150
N_JOINTS    = 6
IN_CHANNELS = 3
MIN_FRAMES  = 20
FPS         = 30
N_EPOCHS    = 60
EPS         = 1e-6

# ── Joint order ───────────────────────────────────────────────────────
JOINT_ORDER = ['l_hip', 'l_knee', 'l_ankle', 'r_hip', 'r_knee', 'r_ankle']

# ── Exercise list ─────────────────────────────────────────────────────
# SINGLE definition — RehabNet._N_EXERCISES must equal len(EXERCISE_LIST)
EXERCISE_LIST = [
    'deep_squat', 'hurdle_step', 'inline_lunge', 'side_lunge',
    'sit_to_stand', 'straight_leg_raise',
    'squat', 'ctk_squat', 'knee_bend', 'unknown'
]
EXERCISE2IDX = {e: i for i, e in enumerate(EXERCISE_LIST)}
N_EXERCISES  = len(EXERCISE_LIST)   # 10  ← matches RehabNet._N_EXERCISES below

# ── Hardware modes ────────────────────────────────────────────────────
HARDWARE_MODES = {
    0: {'name': 'assist', 'scale': -0.3, 'bias': 0.5, 'min': 0.0, 'max': 0.8},
    1: {'name': 'resist', 'scale':  0.6, 'bias': 0.1, 'min': 0.0, 'max': 0.8},
}
N_MODES = len(HARDWARE_MODES)   # 2

# ── Rep segmentation: minimum inter-peak distance per exercise ─────────
EX_MIN_DIST = {
    'inline_lunge': 100,
    'side_lunge':   100,
    'deep_squat':    60,
    'sit_to_stand':  80,
    'straight_leg_raise': 40,
    'knee_bend':     60,
    'squat':         60,
    'ctk_squat':     60,
    'hurdle_step':   45,
    'unknown':       60,
    'default':       60,
}

# ── Deficit targets per exercise (degrees) ────────────────────────────
TARGET_ANGLES = {
    'deep_squat': 90., 'inline_lunge': 90., 'side_lunge': 90.,
    'sit_to_stand': 100., 'straight_leg_raise': 170.,
    'knee_bend': 90., 'squat': 90., 'default': 90.,
}

# ── Feedback rules (priority-ordered, highest = most important) ────────
# scalars index map:
#   0=l_rom/180  1=r_rom/180  2=l_peak/180  3=r_peak/180
#   4=sym/100    5=vel/200    6=jerk/10     7=trunk
#   8=lag/150    9=smooth
FEEDBACK_RULES = [
    {'name': 'insufficient_ROM',   'priority': 10,
     'check':   lambda s: min(s[0], s[1]) * 180 < 50,
     'message': 'Bend your knee a little further — try to reach 90°.',
     'short':   'Bend knee further'},
    {'name': 'too_fast',           'priority': 9,
     'check':   lambda s: s[5] * 200 > 80,
     'message': 'Slow down — take about 3 seconds for each bend.',
     'short':   'Move slower'},
    {'name': 'high_jerk',          'priority': 8,
     'check':   lambda s: s[6] * 10 > 8,
     'message': "Don't jerk your knee — keep the movement smooth.",
     'short':   'Avoid jerking'},
    {'name': 'trunk_compensation', 'priority': 7,
     'check':   lambda s: s[7] > 0.35,
     'message': 'Keep your back straight — try not to lean forward.',
     'short':   'Keep back straight'},
    {'name': 'asymmetric',         'priority': 6,
     'check':   lambda s: s[4] * 100 > 25,
     'message': 'Try to put equal weight on both legs.',
     'short':   'Equal weight both legs'},
    {'name': 'temporal_lag',       'priority': 5,
     'check':   lambda s: s[8] * 150 > 15,
     'message': 'Keep both legs moving at the same time.',
     'short':   'Sync both legs'},
]

# ── MediaPipe model path ───────────────────────────────────────────────
MP_MODEL_PATH = '/tmp/pose_landmarker.task'
MP_MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/'
                 'pose_landmarker/pose_landmarker_heavy/float16/'
                 'latest/pose_landmarker_heavy.task')

print(f'Constants loaded. N_EXERCISES={N_EXERCISES}  N_MODES={N_MODES}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab — Drive mounted
Device: cuda
Constants loaded. N_EXERCISES=10  N_MODES=2


In [ ]:
def ps1_bone_normalize(df):
    for side, (h, k, a) in [
        ('L', ('HipLeft',  'KneeLeft',  'AnkleLeft')),
        ('R', ('HipRight', 'KneeRight', 'AnkleRight')),
    ]:
        leg_len = np.sqrt(
            (df[f'{h}_x'] - df[f'{a}_x'])**2 +
            (df[f'{h}_y'] - df[f'{a}_y'])**2 +
            (df[f'{h}_z'] - df[f'{a}_z'])**2
        ).mean() + 1e-6
        for jname in [h, k, a]:
            for ax in ['x', 'y', 'z']:
                df[f'{jname}_{ax}'] = df[f'{jname}_{ax}'] / leg_len
    return df

In [ ]:
# ─────────────────────────────────────────────────────────────────
# A. SHARED HELPERS
# ─────────────────────────────────────────────────────────────────

def safe_np(x, fill=0.0, clip=None, dtype=np.float32):
    x = np.asarray(x, dtype=dtype)
    x = np.nan_to_num(x, nan=fill, posinf=fill, neginf=fill)
    if clip is not None:
        x = np.clip(x, -clip, clip)
    return x.astype(dtype)

def valid_np(x):
    x = np.asarray(x)
    return x.size > 0 and np.isfinite(x).all()

def _smooth(arr, win=7, poly=2):
    arr = safe_np(arr)
    if len(arr) < win + 2:
        return arr
    w = win if win % 2 == 1 else win + 1
    w = max(w, poly + 2 if (poly + 2) % 2 == 1 else poly + 3)
    try:
        return safe_np(savgol_filter(arr, w, poly))
    except Exception:
        return arr

def _resample(arr, n):
    arr = safe_np(arr)
    if len(arr) == n:
        return arr
    if len(arr) < 2:
        return np.zeros(n, dtype=np.float32)
    return safe_np(np.interp(np.linspace(0, 1, n),
                              np.linspace(0, 1, len(arr)), arr))

def _fill_missing(kps):
    kps = safe_np(kps)
    T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:, j, c]
            z   = (~np.isfinite(col)) | (col == 0.0)
            if z.all():
                opp = j + 3 if j < 3 else j - 3
                kps[:, j, c] = kps[:, opp, c]
            elif z.any() and (~z).sum() >= 2:
                f = interp1d(np.where(~z)[0], col[~z],
                             bounds_error=False, fill_value='extrapolate')
                kps[:, j, c] = f(np.arange(T))
    return safe_np(kps)

def _normalise_kps(kps):
    kps  = safe_np(kps)
    hip  = ((kps[:, 0, :] + kps[:, 3, :]) / 2.0).mean(axis=0)
    kps  = kps - hip
    l    = np.mean(np.linalg.norm(kps[:, 0, :] - kps[:, 2, :], axis=1))
    r    = np.mean(np.linalg.norm(kps[:, 3, :] - kps[:, 5, :], axis=1))
    scale = (l + r) / 2.0
    if not np.isfinite(scale) or scale < 1e-4:
        scale = 1.0
    return safe_np(np.clip(kps / scale, -5.0, 5.0), clip=5.0)

def build_adjacency():
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)
    for i, j in [(0, 1), (1, 2), (3, 4), (4, 5), (0, 3), (1, 4), (2, 5)]:
        A[i, j] = A[j, i] = 1.0
    np.fill_diagonal(A, 1.0)
    d = np.diag(1.0 / np.sqrt(A.sum(axis=1) + EPS))
    return (d @ A @ d).astype(np.float32)

ADJ = build_adjacency()

# ─────────────────────────────────────────────────────────────────
# B. PS1 — VIDEO → DATAFRAME
# ─────────────────────────────────────────────────────────────────

MIN_DETECT_CONF = 0.5
MIN_TRACK_CONF  = 0.5

JOINT_COLS = []
for _jn in ['HipLeft', 'KneeLeft', 'AnkleLeft', 'HipRight', 'KneeRight', 'AnkleRight']:
    for _ax in ['x', 'y', 'z']:
        JOINT_COLS.append(f'{_jn}_{_ax}')
BASE_COLS = ['subject_id', 'session', 'label', 'frame_id', 'time_s', 'detection']

def ps1_prepare_frame(frame, target_w=640):
    h, w = frame.shape[:2]
    if w > target_w:
        frame = cv2.resize(frame, (target_w, int(h * target_w / w)))
    return frame

def ps1_extract_row(landmarks):
    mp_map = {
        'HipLeft': 23, 'KneeLeft': 25, 'AnkleLeft': 27,
        'HipRight': 24, 'KneeRight': 26, 'AnkleRight': 28,
    }
    row = {}
    for jname, idx in mp_map.items():
        lm = landmarks[idx]
        row[f'{jname}_x'] = float(lm.x)
        row[f'{jname}_y'] = float(lm.y)
        row[f'{jname}_z'] = float(lm.z)
    return row

def ps1_root_center(df):
    for ax in ['x', 'y', 'z']:
        mid = (df[f'HipLeft_{ax}'] + df[f'HipRight_{ax}']) / 2.0
        for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
                   'HipRight', 'KneeRight', 'AnkleRight']:
            df[f'{jn}_{ax}'] = df[f'{jn}_{ax}'] - mid
    return df

def ps1_bone_normalize(df):
    for side, (h, k, a) in [
        ('L', ('HipLeft',  'KneeLeft',  'AnkleLeft')),
        ('R', ('HipRight', 'KneeRight', 'AnkleRight')),
    ]:
        for ax in ['x', 'y', 'z']:
            thigh = np.sqrt(((df[f'{h}_{ax}'] - df[f'{k}_{ax}']) ** 2).mean()) + EPS
            for jname in [h, k, a]:
                df[f'{jname}_{ax}'] = df[f'{jname}_{ax}'] / thigh
    return df

def _knee_angle_from_df(df, side='L'):
    prefix = {
        'L': ('HipLeft',  'KneeLeft',  'AnkleLeft'),
        'R': ('HipRight', 'KneeRight', 'AnkleRight'),
    }[side]
    h  = df[[f'{prefix[0]}_{ax}' for ax in 'xyz']].values
    k  = df[[f'{prefix[1]}_{ax}' for ax in 'xyz']].values
    a  = df[[f'{prefix[2]}_{ax}' for ax in 'xyz']].values
    v1 = h - k
    v2 = a - k
    cos = (np.sum(v1 * v2, axis=1) /
           (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + EPS))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def ps1_extract_features(df):
    df['angle_knee_L']        = _knee_angle_from_df(df, 'L')
    df['angle_knee_R']        = _knee_angle_from_df(df, 'R')
    df['angle_knee_L_smooth'] = _smooth(df['angle_knee_L'].values)
    df['angle_knee_R_smooth'] = _smooth(df['angle_knee_R'].values)
    return df

def ps1_clean_nulls(df):
    return df.ffill().bfill().fillna(0.0)

def _ensure_mp_model():
    if not os.path.exists(MP_MODEL_PATH):
        print('  Downloading MediaPipe pose model (~30 MB)...')
        urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)
        print('  Downloaded.')

def ps1_process_video(video_path, patient_id,
                      exercise='unknown', session='1', label='unknown'):
    """
    VIDEO FILE → processed DataFrame  (one row per detected frame).
    Works on any mp4/avi/webm file.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open video: {video_path}')

    fps_vid    = cap.get(cv2.CAP_PROP_FPS) or FPS
    rows, frame_idx, detected = [], 0, 0
    t0 = time.time()

    _ensure_mp_model()

    import mediapipe as mp
    from mediapipe.tasks import python as _mptasks
    from mediapipe.tasks.python import vision as _mpvision

    opts = _mpvision.PoseLandmarkerOptions(
        base_options=_mptasks.BaseOptions(model_asset_path=MP_MODEL_PATH),
        running_mode=_mpvision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=MIN_DETECT_CONF,
        min_pose_presence_confidence=MIN_DETECT_CONF,
        min_tracking_confidence=MIN_TRACK_CONF,
        output_segmentation_masks=False,
    )

    frame_step_ms = int(1000.0 / fps_vid)

    with _mpvision.PoseLandmarker.create_from_options(opts) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_idx += 1
            frame_ms   = frame_idx * frame_step_ms
            prepped    = ps1_prepare_frame(frame)
            rgb        = cv2.cvtColor(prepped, cv2.COLOR_BGR2RGB)
            mp_img     = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res        = pose.detect_for_video(mp_img, frame_ms)

            row = {'subject_id': patient_id, 'session': session,
                   'label': label, 'frame_id': frame_idx,
                   'time_s': round(frame_idx / fps_vid, 4), 'detection': 0}
            row.update({col: float('nan') for col in JOINT_COLS})

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                detected += 1
                row['detection'] = 1
                row.update(ps1_extract_row(res.pose_landmarks[0]))
            rows.append(row)

    cap.release()

    if frame_idx == 0:
        raise ValueError('Video is empty')

    det_pct = 100 * detected / frame_idx
    print(f'  PS1: {frame_idx} frames  detection={det_pct:.0f}%  '
          f'{time.time()-t0:.1f}s')
    if det_pct < 20:
        print(f'  WARNING: low detection ({det_pct:.0f}%) — '
              f'check lighting and camera angle')

    df = pd.DataFrame(rows, columns=BASE_COLS + JOINT_COLS)
    df = df[df['detection'] == 1].reset_index(drop=True)
    df = ps1_root_center(df)
    df = ps1_bone_normalize(df)
    df = ps1_extract_features(df)
    df = ps1_clean_nulls(df)
    return df


# ─────────────────────────────────────────────────────────────────
# C. KEYPOINT PROCESSING  (shared by PS1 pipeline and dataset loaders)
# ─────────────────────────────────────────────────────────────────

def process_keypoints(raw):
    """
    raw: np.ndarray shape (T, N_JOINTS, 3)
    Returns: np.ndarray shape (TARGET_LEN, N_JOINTS, 3), normalised
    """
    raw = safe_np(raw)
    if raw.ndim != 3 or raw.shape[1] != N_JOINTS or raw.shape[2] != 3:
        raise ValueError(f'Expected (T,{N_JOINTS},3), got {raw.shape}')
    if len(raw) < MIN_FRAMES:
        raise ValueError(f'Too short: {len(raw)} frames (min={MIN_FRAMES})')
    for j in range(N_JOINTS):
        for c in range(3):
            raw[:, j, c] = _smooth(raw[:, j, c])
    raw = _fill_missing(raw)
    out = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
    for j in range(N_JOINTS):
        for c in range(3):
            out[:, j, c] = _resample(raw[:, j, c], TARGET_LEN)
    return safe_np(_normalise_kps(out), clip=5.0)

def extract_scalars(kps):
    """
    kps: np.ndarray (TARGET_LEN, N_JOINTS, 3)
    Returns: np.ndarray (10,) normalised scalar features
    """
    kps = safe_np(kps, clip=5.0)

    def _ang(a, b, c):
        ba, bc = a - b, c - b
        d = np.maximum(np.linalg.norm(ba, 1) * np.linalg.norm(bc, 1), EPS)
        return np.degrees(np.arccos(np.clip(np.sum(ba * bc, 1) / d, -1, 1)))

    lk  = _ang(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    rk  = _ang(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])
    lr  = float(np.nanmax(lk) - np.nanmin(lk))
    rr  = float(np.nanmax(rk) - np.nanmin(rk))
    sym = abs(lr - rr) / ((lr + rr) / 2 + EPS) * 100 if (lr + rr) > 1 else 0.0
    vel = float(np.max(np.abs(np.diff(lk))) * FPS) if len(lk) > 1 else 0.0

    hip  = (kps[:, 0, :] + kps[:, 3, :]) / 2.0
    jk   = np.diff(np.diff(hip, axis=0), axis=0)
    jerk = float(np.nanmean(np.linalg.norm(jk, axis=1))) if len(jk) else 0.0

    pk    = int(np.argmin(lk))
    trunk = float(abs(hip[pk, 0] - hip[0, 0]))

    lkc, rkc = lk - np.nanmean(lk), rk - np.nanmean(rk)
    lag = float(abs(
        np.argmax(np.correlate(lkc, rkc, 'full')) - (len(lk) - 1)
    )) if np.nanstd(lkc) > EPS and np.nanstd(rkc) > EPS else 0.0

    smooth = float(np.clip(1.0 / (np.nanstd(np.diff(lk)) + EPS) / 100, 0, 1))

    return safe_np(np.array([
        np.clip(lr / 180,          0, 1),
        np.clip(rr / 180,          0, 1),
        np.clip(np.min(lk) / 180,  0, 1),
        np.clip(np.min(rk) / 180,  0, 1),
        np.clip(sym / 100,         0, 5),
        np.clip(vel / 200,         0, 5),
        np.clip(jerk / 10,         0, 5),
        np.clip(trunk,             0, 5),
        np.clip(lag / TARGET_LEN,  0, 1),
        smooth,
    ], dtype=np.float32), fill=0.0, clip=10.0)


# ─────────────────────────────────────────────────────────────────
# D. BRIDGE — DATAFRAME → PER-REP SAMPLE DICTS
# ─────────────────────────────────────────────────────────────────

def _df_to_kps(df):
    cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    return df[cols].values.astype(np.float32).reshape(len(df), N_JOINTS, 3)

def _segment_reps(df, angle_col='angle_knee_L_smooth',
                  min_frames=MIN_FRAMES, min_rom=20.0, exercise='unknown'):
    """Split a continuous DataFrame into per-rep DataFrames."""
    if angle_col in df.columns:
        angles = df[angle_col].ffill().bfill().values
    else:
        angles = _knee_angle_from_df(df, 'L')

    n = len(angles)
    ex_key = exercise if exercise in EX_MIN_DIST else 'default'
    min_dist = max(EX_MIN_DIST.get(ex_key, EX_MIN_DIST['default']), n // 20)
    width = max(3, min_dist // 4)

    peaks, _ = find_peaks(
        -angles,
        distance=min_dist,
        prominence=min_rom * 0.5,
        width=width,
    )

    print(f'  Segmenter: exercise={exercise}  n_frames={n}  '
          f'min_dist={min_dist}  peaks={len(peaks)}')

    if len(peaks) == 0:
        return [df]

    mids   = [(peaks[i] + peaks[i + 1]) // 2 for i in range(len(peaks) - 1)]
    bounds = [0] + mids + [n]
    reps   = [df.iloc[bounds[i]:bounds[i + 1]].copy()
              for i in range(len(bounds) - 1)
              if len(df.iloc[bounds[i]:bounds[i + 1]]) >= min_frames]
    return reps if reps else [df]

def ps1_df_to_samples(df, patient_id, exercise='unknown',
                       label=-1, source='patient_video'):
    """PS1 DataFrame → list of sample dicts ready for RehabNet."""
    reps   = _segment_reps(df, exercise=exercise)
    ex_idx = EXERCISE2IDX.get(exercise, EXERCISE2IDX['unknown'])
    print(f'  Bridge: {len(reps)} rep segments → ', end='')
    samples = []
    for i, rep_df in enumerate(reps):
        try:
            kps = _df_to_kps(rep_df)
            kps = process_keypoints(kps)
            sc  = extract_scalars(kps)
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'scalars':      sc,
                'label':        max(label, 0),
                'quality':      float(max(label, 0)),
                'exercise':     exercise,
                'exercise_idx': ex_idx,
                'subject':      patient_id,
                'source':       source,
                'rep_id':       f'{patient_id}_r{i + 1}',
            })
        except Exception as e:
            print(f'\n    Rep {i + 1} skipped: {e}')
    print(f'{len(samples)} valid')
    return samples


# ─────────────────────────────────────────────────────────────────
# E. DATASET LOADERS
# ─────────────────────────────────────────────────────────────────

def _kps_from_df_rows(df, xyz_cols):
    return (df[xyz_cols].values
            .astype(np.float32)
            .reshape(len(df), N_JOINTS, 3))

def load_uiprmd():
    if not os.path.exists(UIPRMD_CSV):
        print(f'UI-PRMD not found: {UIPRMD_CSV}')
        return []
    df       = pd.read_csv(UIPRMD_CSV)
    xyz_cols = [f'{j}_{ax}' for j in JOINT_ORDER for ax in ['x', 'y', 'z']]
    samples  = []
    for (subj, mov, ex_id, lv), grp in df.groupby(
            ['subject', 'movement', 'exercise_id', 'label']):
        grp = grp.sort_values('frame')
        raw = _kps_from_df_rows(grp, xyz_cols) / 1000.0
        try:
            kps = process_keypoints(raw)
            ex  = grp['exercise'].iloc[0]
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(lv),
                'quality':      float(lv),
                'exercise':     ex,
                'exercise_idx': EXERCISE2IDX.get(ex, EXERCISE2IDX['unknown']),
                'subject':      f'uiprmd_{subj}',
                'source':       'uiprmd',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip uiprmd {subj}/{mov}/{ex_id}: {e}')
    print(f'UI-PRMD: {len(samples)} samples')
    return samples

def load_kimore():
    if not os.path.exists(KIMORE_CSV):
        print(f'KIMORE not found: {KIMORE_CSV}')
        return []
    df = pd.read_csv(KIMORE_CSV)
    xyz_cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    df['quality_norm'] = df[['cTS', 'cPO', 'cCF']].mean(axis=1) / 50.0
    samples = []
    for subj, grp in df.groupby('Subject'):
        grp = grp.sort_values('FrameID')
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps = process_keypoints(raw)
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(grp['label'].iloc[0]),
                'quality':      float(grp['quality_norm'].mean()),
                'exercise':     'squat',
                'exercise_idx': EXERCISE2IDX['squat'],
                'subject':      f'kimore_{subj}',
                'source':       'kimore',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip kimore {subj}: {e}')
    print(f'KIMORE: {len(samples)} samples')
    return samples

def load_keraal():
    if not os.path.exists(KERAAL_CSV):
        print(f'Keraal not found: {KERAAL_CSV}')
        return []
    df = pd.read_csv(KERAAL_CSV)
    xyz_cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    samples = []
    for ann_key, grp in df.groupby('ann_key'):
        grp = grp.sort_values('FrameID') if 'FrameID' in grp.columns else grp
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps  = process_keypoints(raw)
            subj = '-'.join(str(ann_key).split('-')[:3])
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(grp['label'].iloc[0]),
                'quality':      float(grp['label'].iloc[0]),
                'exercise':     'ctk_squat',
                'exercise_idx': EXERCISE2IDX['ctk_squat'],
                'subject':      f'keraal_{subj}',
                'source':       'keraal',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip keraal {ann_key}: {e}')
    print(f'Keraal: {len(samples)} samples')
    return samples

def build_master_dataset():
    raw   = load_uiprmd() + load_kimore() + load_keraal()
    clean = []
    for s in raw:
        try:
            kps = safe_np(s['keypoints'], clip=5.0)
            sc  = safe_np(s.get('scalars', extract_scalars(kps)), clip=10.0)
            q   = float(np.nan_to_num(s.get('quality', s.get('label', 1)),
                                      nan=float(s.get('label', 1))))
            ex  = int(s.get('exercise_idx', EXERCISE2IDX['unknown']))
            if kps.shape != (TARGET_LEN, N_JOINTS, 3):
                raise ValueError('bad kps shape')
            if sc.shape != (10,):
                raise ValueError('bad scalar shape')
            if not valid_np(kps) or not valid_np(sc):
                raise ValueError('NaN/Inf detected')
            if not (0 <= ex < N_EXERCISES):
                ex = EXERCISE2IDX['unknown']
            s.update({'keypoints': kps, 'scalars': sc,
                      'label':        int(s.get('label', 1)),
                      'quality':      float(np.clip(q, 0, 1)),
                      'exercise_idx': ex})
            clean.append(s)
        except Exception as e:
            print(f"  DROP {s.get('subject', '?')}: {e}")
    print(f'\nMaster dataset: {len(clean)} samples')
    print(f"  correct={sum(s['label']==1 for s in clean)}  "
          f"incorrect={sum(s['label']==0 for s in clean)}")
    print(f"  sources:   {set(s['source']   for s in clean)}")
    print(f"  exercises: {set(s['exercise'] for s in clean)}")
    return clean

def loso_split(samples, held_out):
    return ([s for s in samples if s['subject'] != held_out],
            [s for s in samples if s['subject'] == held_out])

def get_uiprmd_subjects(samples):
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})


# ─────────────────────────────────────────────────────────────────
# F. MODEL — STGCNBlock → STGCN → ClinicalTransformer → RehabNet
# ─────────────────────────────────────────────────────────────────

class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, ks=9, stride=1):
        super().__init__()
        self.gcn = nn.Linear(c_in, c_out)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, (ks, 1), (stride, 1), ((ks - 1) // 2, 0)),
            nn.BatchNorm2d(c_out), nn.Dropout(0.1),
        )
        self.res = (nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride, 1)),
                                   nn.BatchNorm2d(c_out))
                    if c_in != c_out or stride != 1 else nn.Identity())
        self.relu = nn.ReLU()

    def forward(self, x, A):
        B, C, T, V = x.shape
        if A.dim() == 3:
            A = A[0]
        xs = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)
        xs = torch.bmm(A.unsqueeze(0).expand(B * T, -1, -1), xs)
        xs = self.gcn(xs).view(B, T, V, -1).permute(0, 3, 1, 2)
        return self.relu(
            torch.nan_to_num(self.tcn(xs), 0.0) +
            torch.nan_to_num(self.res(x),  0.0)
        )


class STGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(IN_CHANNELS * N_JOINTS)
        self.b1 = STGCNBlock(3, 32)
        self.b2 = STGCNBlock(32, 64)
        self.b3 = STGCNBlock(64, 128)

    def forward(self, x, A):
        B, C, T, V = x.shape
        x  = torch.clamp(x, -5.0, 5.0)
        xf = x.permute(0, 2, 1, 3).contiguous().view(B * T, C * V)
        xf = self.input_norm(xf).view(B, T, C, V).permute(0, 2, 1, 3)
        return self.b3(self.b2(self.b1(xf, A), A), A).mean(3).permute(0, 2, 1)


class ClinicalTransformer(nn.Module):
    def __init__(self, d=128, heads=4, d_ff=256, drop=0.1):
        super().__init__()
        pe  = torch.zeros(TARGET_LEN + 1, d)
        pos = torch.arange(TARGET_LEN + 1).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-np.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.cls  = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.drop = nn.Dropout(drop)
        mk = lambda: nn.TransformerEncoderLayer(
            d, heads, d_ff, drop, batch_first=True, norm_first=True)
        self.l1, self.l2 = mk(), mk()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, return_attn=False):
        B, T, _ = x.shape
        x = torch.cat([self.cls.expand(B, -1, -1), x], 1)
        x = self.drop(x + self.pe[:, :T + 1, :])
        x = self.n1(self.l1(x))
        if return_attn:
            sa  = self.l2.self_attn
            xn  = self.l2.norm1(x)
            out, attn = sa(xn, xn, xn, need_weights=True,
                           average_attn_weights=True)
            x2  = x + self.l2.dropout1(out)
            xn2 = self.l2.norm2(x2)
            ff  = self.l2.linear2(
                self.l2.dropout(self.l2.activation(self.l2.linear1(xn2)))
            )
            return self.n2(x2 + self.l2.dropout2(ff))[:, 0, :], attn
        return self.n2(self.l2(x))[:, 0, :], None


class RehabNet(nn.Module):
    # These must match the constants in Cell 2.
    # If you change N_EXERCISES or N_MODES there, update these too.
    _N_MODES     = 2    # ← must equal N_MODES
    _N_EXERCISES = 10   # ← must equal N_EXERCISES  (FIX: was 9)

    def __init__(self):
        super().__init__()
        self.stgcn       = STGCN()
        self.trans       = ClinicalTransformer()
        self.bilstm      = nn.LSTM(128, 64, num_layers=2, batch_first=True,
                                   bidirectional=True, dropout=0.1)
        self.bilstm_proj = nn.Sequential(
            nn.Linear(128, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.1))
        self.mode_emb    = nn.Embedding(self._N_MODES,      16)
        self.ex_emb      = nn.Embedding(self._N_EXERCISES,   8)
        # 128 (cls) + 128 (bilstm) + 10 (scalars) + 16 (mode) + 8 (ex) = 290
        self.shared      = nn.Sequential(
            nn.Linear(290, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2))
        self.classify_head = nn.Linear(128, 2)
        self.quality_head  = nn.Linear(128, 1)
        self.ex_head       = nn.Linear(128, self._N_EXERCISES)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode(self, kps, adj, return_attn=False):
        if adj.dim() == 3:
            adj = adj[0]
        feats        = self.stgcn(kps, adj)
        cls, attn    = self.trans(feats, return_attn)
        lstm_out, _  = self.bilstm(feats)
        bilstm_ctx   = self.bilstm_proj(lstm_out.mean(1))
        return cls, bilstm_ctx, attn

    def forward(self, kps, adj, scalars, mode=None, exercise_idx=None):
        B, device = kps.shape[0], kps.device
        if mode         is None: mode         = torch.zeros(B, dtype=torch.long, device=device)
        if exercise_idx is None: exercise_idx = torch.zeros(B, dtype=torch.long, device=device)
        mode         = torch.clamp(mode,         0, self._N_MODES     - 1)
        exercise_idx = torch.clamp(exercise_idx, 0, self._N_EXERCISES - 1)
        cls, bilstm_ctx, _ = self.encode(kps, adj)
        x = self.shared(torch.cat([
            cls, bilstm_ctx, scalars,
            self.mode_emb(mode),
            self.ex_emb(exercise_idx),
        ], dim=1))
        # Returns 3 values — always unpack with *_ if you only need 2
        return (self.classify_head(x),
                self.quality_head(x).squeeze(1),
                self.ex_head(cls))

    def predict_exercise(self, kps, adj):
        self.eval()
        with torch.no_grad():
            if adj.dim() == 3:
                adj = adj[0]
            cls, _, _ = self.encode(kps, adj)
            idx = int(self.ex_head(cls).argmax(1).item())
        return EXERCISE_LIST[idx], idx


# ─────────────────────────────────────────────────────────────────
# G. PYTORCH DATASET + DATALOADER HELPERS
# ─────────────────────────────────────────────────────────────────

class RehabDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = safe_np(s['keypoints'], clip=5.0)
        if self.augment:
            kps = self._aug(kps)
        return {
            'keypoints':    torch.from_numpy(safe_np(kps, clip=5.0)).permute(2, 0, 1).float(),
            'adj':          torch.from_numpy(ADJ).float(),
            'label':        torch.tensor(s['label'],        dtype=torch.long),
            'quality':      torch.tensor(s['quality'],      dtype=torch.float32),
            'scalars':      torch.from_numpy(safe_np(s['scalars'], clip=10.0)).float(),
            'exercise_idx': torch.tensor(s['exercise_idx'], dtype=torch.long),
            'exercise':     s['exercise'],
            'subject':      s['subject'],
        }

    def _aug(self, kps):
        kps = safe_np(kps, clip=5.0)
        if np.random.rand() < 0.5:  # temporal jitter ±20%
            n   = max(int(TARGET_LEN * np.random.uniform(0.8, 1.2)), MIN_FRAMES)
            tmp = np.zeros((n, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    tmp[:, j, c] = _resample(kps[:, j, c], n)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    kps[:, j, c] = _resample(tmp[:, j, c], TARGET_LEN)
        if np.random.rand() < 0.5:  # mirror flip L↔R
            kps[:, [0, 1, 2, 3, 4, 5], :] = kps[:, [3, 4, 5, 0, 1, 2], :]
            kps[:, :, 0] *= -1.0
        if np.random.rand() < 0.5:  # Gaussian noise
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02
        if np.random.rand() < 0.3:  # small rotation in XZ plane
            th  = np.radians(np.random.uniform(-10, 10))
            x, z = kps[:, :, 0].copy(), kps[:, :, 2].copy()
            kps[:, :, 0] = np.cos(th) * x - np.sin(th) * z
            kps[:, :, 2] = np.sin(th) * x + np.cos(th) * z
        return safe_np(kps, clip=5.0)


def collate(batch):
    return {
        k: torch.stack([b[k] for b in batch])
           if isinstance(batch[0][k], torch.Tensor)
           else [b[k] for b in batch]
        for k in batch[0]
    }

def build_loaders(train_samples, test_samples, batch_size=32):
    """Build train and validation DataLoaders with exercise-balanced sampling."""
    ex_counts = Counter(s['exercise'] for s in train_samples)
    total     = len(train_samples)
    weights   = [total / ex_counts[s['exercise']] for s in train_samples]
    sampler   = WeightedRandomSampler(weights, len(train_samples), replacement=True)
    train_loader = DataLoader(
        RehabDataset(train_samples, augment=True),
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=collate,
        num_workers=2,
    )
    val_loader = DataLoader(
        RehabDataset(test_samples, augment=False),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
    )
    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────
# H. TRAINING
# ─────────────────────────────────────────────────────────────────

def compute_class_weights(train_samples, device):
    """Inverse-frequency weights for cross-entropy loss."""
    n0  = sum(s['label'] == 0 for s in train_samples)
    n1  = sum(s['label'] == 1 for s in train_samples)
    tot = n0 + n1
    return torch.tensor(
        [tot / (2 * n0 + EPS), tot / (2 * n1 + EPS)],
        dtype=torch.float32, device=device,
    )

def _train_epoch(model, loader, opt, device, cw, epoch=0):
    model.train()
    A = torch.from_numpy(ADJ).to(device)
    total, used, skipped = 0.0, 0, 0

    for b in loader:
        kps    = b['keypoints'].to(device)
        lbl    = b['label'].to(device)
        qual   = b['quality'].to(device)
        sc     = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)

        good = (torch.isfinite(kps).flatten(1).all(1) &
                torch.isfinite(sc).all(1) &
                torch.isfinite(qual))
        if not good.all():
            kps, lbl, qual, sc, ex_idx = (t[good] for t in
                                           (kps, lbl, qual, sc, ex_idx))
        if kps.shape[0] == 0:
            skipped += 1
            continue

        mode = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        lg, qp, ex_lg = model(kps, A, sc, mode, ex_idx)

        if not (torch.isfinite(lg).all() and torch.isfinite(qp).all()):
            skipped += 1
            continue

        loss = (F.cross_entropy(lg, lbl, weight=cw) +
                0.3 * F.mse_loss(torch.sigmoid(qp),
                                  torch.clamp(qual, 0, 1)) +
                0.5 * F.cross_entropy(ex_lg, ex_idx))

        if not torch.isfinite(loss):
            skipped += 1
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
        total += loss.item()
        used  += 1

    if skipped:
        print(f'  ep{epoch}: skipped {skipped} batches')
    return total / max(used, 1)

@torch.no_grad()
def _evaluate(model, loader, device):
    model.eval()
    A = torch.from_numpy(ADJ).to(device)
    preds, labels, quals, qpreds = [], [], [], []

    for b in loader:
        kps    = b['keypoints'].to(device)
        sc     = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)
        qual   = b['quality'].to(device)
        lbl    = b['label'].to(device)

        good = (torch.isfinite(kps).flatten(1).all(1) &
                torch.isfinite(sc).all(1))
        if good.sum() == 0:
            continue
        kps, sc, ex_idx, qual, lbl = (t[good] for t in
                                       (kps, sc, ex_idx, qual, lbl))
        mode = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        lg, qp, _ = model(kps, A, sc, mode, ex_idx)  # unpack all 3

        preds.extend(lg.argmax(1).cpu().numpy())
        labels.extend(lbl.cpu().numpy())
        quals.extend(qual.cpu().numpy())
        qpreds.extend(torch.sigmoid(qp).cpu().numpy())

    if not labels:
        return {'acc': 0.0, 'f1': 0.0, 'quality_r': 0.0}

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    r   = pearsonr(quals, qpreds)[0] if len(set(quals)) > 1 else 0.0
    print(classification_report(labels, preds,
          target_names=['incorrect', 'correct'], zero_division=0))
    print(f'  quality Pearson r={r:.3f}')
    return {'acc': acc, 'f1': f1, 'quality_r': r}

def run_loso(samples=None, n_epochs=N_EPOCHS,
             batch_size=32, lr=3e-4, patience=12):
    """
    Leave-One-Subject-Out cross-validation.

    Usage:
        # Option A — loads datasets internally
        results, model = run_loso()

        # Option B — pass your own sample list
        results, model = run_loso(samples=my_samples)
    """
    if samples is None:
        samples = build_master_dataset()

    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    subjects = get_uiprmd_subjects(samples)
    if not subjects:
        subjects = sorted({s['subject'] for s in samples})

    results = []
    best_overall_f1    = 0.0
    best_overall_model = None

    for subj in subjects:
        print(f"\n{'='*55}\nLOSO held-out: {subj}\n{'='*55}")
        tr, te = loso_split(samples, subj)
        tl, vl = build_loaders(tr, te, batch_size)
        cw     = compute_class_weights(tr, device)
        print(f'  class weights: incorrect={cw[0]:.3f}  correct={cw[1]:.3f}')

        model = RehabNet().to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        def lr_fn(ep):
            w = 5
            if ep < w:
                return (ep + 1) / w
            return 0.5 * (1 + np.cos(np.pi * (ep - w) / max(n_epochs - w, 1)))

        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
        best_f1, best_state, no_imp = 0.0, None, 0

        for ep in range(n_epochs):
            loss = _train_epoch(model, tl, opt, device, cw, epoch=ep)
            sch.step()

            if (ep + 1) % 5 == 0:
                model.eval()
                pl, ll = [], []
                with torch.no_grad():
                    A2 = torch.from_numpy(ADJ).to(device)
                    for b in vl:
                        kps    = b['keypoints'].to(device)
                        sc     = b['scalars'].to(device)
                        ex_idx = b['exercise_idx'].to(device)
                        lbl    = b['label']
                        good   = (torch.isfinite(kps).flatten(1).all(1) &
                                  torch.isfinite(sc).all(1))
                        if good.sum() == 0:
                            continue
                        kps, sc, ex_idx = kps[good], sc[good], ex_idx[good]
                        lbl  = lbl[good.cpu()]
                        mode = torch.zeros(kps.shape[0], dtype=torch.long,
                                           device=device)
                        lg, _, _ = model(kps, A2, sc, mode, ex_idx)
                        pl.extend(lg.argmax(1).cpu().numpy())
                        ll.extend(lbl.numpy())
                model.train()

                vf1 = f1_score(ll, pl, average='macro', zero_division=0)
                print(f'  ep{ep+1:3d} loss={loss:.4f}  '
                      f'lr={sch.get_last_lr()[0]:.2e}  val_f1={vf1:.3f}')

                if vf1 > best_f1:
                    best_f1    = vf1
                    best_state = copy.deepcopy(model.state_dict())
                    no_imp     = 0
                else:
                    no_imp += 1
                    if no_imp >= patience:
                        print(f'  Early stop ep{ep+1}')
                        break

        if best_state:
            model.load_state_dict(best_state)

        if best_f1 > best_overall_f1:
            best_overall_f1    = best_f1
            best_overall_model = copy.deepcopy(model.state_dict())

        print(f'\n── Eval: {subj} (best val_f1={best_f1:.3f}) ──')
        results.append(_evaluate(model, vl, device))

    # Save best checkpoint
    if best_overall_model:
        os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)
        torch.save(best_overall_model, MODEL_SAVE)
        print(f'\nBest model saved → {MODEL_SAVE}')

    accs = [r['acc'] for r in results]
    f1s  = [r['f1']  for r in results]
    print(f"\n{'='*55}")
    print(f'LOSO RESULTS')
    print(f'  Acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}')
    print(f'  F1  = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}')
    print(f"{'='*55}")

    final_model = RehabNet().to(device)
    if best_overall_model:
        final_model.load_state_dict(best_overall_model)
    return results, final_model


# ─────────────────────────────────────────────────────────────────
# I. INFERENCE + FEEDBACK
# ─────────────────────────────────────────────────────────────────

def compute_torque(quality, mode):
    """Single definition — FIX: was defined twice with different logic."""
    m = HARDWARE_MODES.get(mode, HARDWARE_MODES[0])
    return float(np.clip(m['scale'] * quality + m['bias'], m['min'], m['max']))

def generate_feedback(label, quality, scalars):
    """Priority-ordered rule engine → feedback dict."""
    s = np.asarray(scalars)
    triggered = sorted(
        [r for r in FEEDBACK_RULES if r['check'](s)],
        key=lambda r: r['priority'], reverse=True,
    )
    flags = [r['name'] for r in triggered]

    if label == 1 and quality >= 0.75:
        return {'message': 'Great rep — keep it up!',
                'short': 'Great rep', 'flags': [], 'severity': 'good'}
    if label == 1 and quality >= 0.5:
        return {'message': 'Good rep — maintain that form.',
                'short': 'Good rep', 'flags': flags, 'severity': 'good'}
    if triggered:
        top = triggered[0]
        return {'message':  top['message'],
                'short':    top['short'],
                'flags':    flags,
                'severity': 'error' if top['priority'] >= 8 else 'warning'}
    return {'message': 'Focus on smooth, controlled movement.',
            'short': 'Keep it smooth', 'flags': [], 'severity': 'warning'}

def build_hardware_payload(sample, quality, scalars, torque, mode):
    kps = sample['keypoints']

    def _ang(a, b, c):
        ba, bc = a - b, c - b
        d = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + EPS
        return np.degrees(np.arccos(np.clip(np.sum(ba * bc, axis=1) / d, -1, 1)))

    lk     = _ang(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    rk     = _ang(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])
    l_peak = float(np.min(lk));  r_peak = float(np.min(rk))
    l_rom  = float(np.max(lk) - l_peak)
    r_rom  = float(np.max(rk) - r_peak)
    ex     = sample.get('exercise', 'unknown')
    target = TARGET_ANGLES.get(ex, TARGET_ANGLES['default'])
    l_def  = max(0., target - l_peak)
    r_def  = max(0., target - r_peak)

    return {
        'torque':   round(torque, 4),
        'mode':     HARDWARE_MODES[mode]['name'],
        'mode_int': mode,
        'joint': {
            'left_knee_peak_deg':  round(l_peak, 2),
            'right_knee_peak_deg': round(r_peak, 2),
            'left_knee_rom_deg':   round(l_rom, 2),
            'right_knee_rom_deg':  round(r_rom, 2),
        },
        'target': {
            'knee_target_deg':   target,
            'left_deficit_deg':  round(l_def, 2),
            'right_deficit_deg': round(r_def, 2),
        },
        'torque_per_degree': round(torque / max(l_def, 1.0), 5),
    }

@torch.no_grad()
def infer_sample(model, sample, device=DEVICE, hardware_mode=0):
    """
    Run RehabNet on one rep.  Returns full result dict.
    FIX: model returns 3 values; always unpack with *_ or explicitly.
    """
    model.eval()
    kps_t  = (torch.from_numpy(sample['keypoints'])
              .permute(2, 0, 1).unsqueeze(0).to(device))
    adj_t  = torch.from_numpy(ADJ).to(device)
    sc_t   = torch.from_numpy(sample['scalars']).unsqueeze(0).to(device)
    mode_t = torch.tensor([hardware_mode], dtype=torch.long, device=device)
    ex_t   = torch.tensor([sample['exercise_idx']], dtype=torch.long,
                           device=device)

    logits, quality, *_ = model(kps_t, adj_t, sc_t, mode_t, ex_t)  # 3 outputs
    label   = int(logits.argmax(1).item())
    quality = float(torch.sigmoid(quality).item())
    torque  = compute_torque(quality, hardware_mode)
    fb      = generate_feedback(label, quality, sample['scalars'])
    hw      = build_hardware_payload(sample, quality, sample['scalars'],
                                     torque, hardware_mode)

    return {
        'rep_id':         sample.get('rep_id', '?'),
        'exercise':       sample.get('exercise', 'unknown'),
        'label':          label,
        'correct':        label == 1,
        'quality_score':  round(quality, 4),
        'feedback':       fb['message'],
        'feedback_short': fb['short'],
        'severity':       fb['severity'],
        'flags':          fb['flags'],
        'hardware':       hw,
        'torque_signal':  torque,
        'mode_name':      HARDWARE_MODES[hardware_mode]['name'],
        'scalars':        sample['scalars'].tolist(),
    }



def _predict_exercise_from_samples(model, samples, device=DEVICE):
    """
    Predict exercise from all rough rep segments and return majority vote.
    This is used only when exercise=None.
    """
    if not samples:
        return 'unknown', EXERCISE2IDX['unknown']

    votes = []
    adj_t = torch.from_numpy(ADJ).to(device)
    model.eval()

    with torch.no_grad():
        for s in samples:
            kps_t = (torch.from_numpy(s['keypoints'])
                     .permute(2, 0, 1).unsqueeze(0).to(device))
            pred_ex, pred_idx = model.predict_exercise(kps_t, adj_t)
            votes.append((pred_ex, pred_idx))

    if not votes:
        return 'unknown', EXERCISE2IDX['unknown']

    counts = Counter(v[0] for v in votes)
    pred_ex = counts.most_common(1)[0][0]
    pred_idx = EXERCISE2IDX.get(pred_ex, EXERCISE2IDX['unknown'])
    print(f'[Exercise Classification] Votes: {dict(counts)}')
    return pred_ex, pred_idx


# ─────────────────────────────────────────────────────────────────
# J. PATIENT SESSION — VIDEO → PER-REP FEEDBACK
# ─────────────────────────────────────────────────────────────────

def run_patient_video(video_path, patient_id,
                      model=None, exercise=None,
                      hardware_mode=0, device=DEVICE, debug=True):
    """
    Full pipeline: video file -> per-rep feedback dict + session summary.

    Important:
      - If exercise is provided, segmentation uses that exercise directly.
      - If exercise=None and model is provided, the pipeline does:
          rough segmentation -> exercise majority vote -> re-segmentation
        so the final rep count uses exercise-specific EX_MIN_DIST.
    """
    print(f"\n{'='*50}\nPatient: {patient_id}\n{'='*50}")

    active_exercise = exercise or 'unknown'
    print('\n[PS1] Processing video...')
    df = ps1_process_video(video_path, patient_id, active_exercise)

    print('\n[Bridge] Segmenting reps...')

    if exercise is None and model is not None:
        rough_samples = ps1_df_to_samples(df, patient_id, 'unknown')
        if not rough_samples:
            print('No valid rough reps found - check video quality')
            return {}

        pred_ex, pred_idx = _predict_exercise_from_samples(
            model, rough_samples, device
        )
        active_exercise = pred_ex
        print(f'\n[Exercise Classification] Predicted majority: {active_exercise}')

        samples = ps1_df_to_samples(df, patient_id, active_exercise)
        for s in samples:
            s['exercise'] = active_exercise
            s['exercise_idx'] = pred_idx
    else:
        print(f'\n[Exercise] Using: {active_exercise}')
        samples = ps1_df_to_samples(df, patient_id, active_exercise)

    if not samples:
        print('No valid reps found - check video quality')
        return {}

    if model is None:
        print('\n[PS2] No model provided - returning PS1 features only')
        return {
            'patient': patient_id,
            'exercise': active_exercise,
            'n_reps': len(samples),
            'ps1_only': True,
            'samples': samples,
        }

    print('\n[PS2] Running RehabNet inference...')
    results = []

    for s in samples:
        out = infer_sample(model, s, device, hardware_mode)
        status = 'CORRECT' if out['label'] == 1 else 'INCORRECT'
        print(f"  {out['rep_id']:20s} | {status} "
              f"| quality={out['quality_score']:.2f} "
              f"| torque={out['torque_signal']:.2f} "
              f"| {out['feedback_short']}")
        results.append(out)

    n_correct = sum(r['label'] == 1 for r in results)
    avg_q = float(np.mean([r['quality_score'] for r in results]))
    avg_t = float(np.mean([r['torque_signal'] for r in results]))

    summary = {
        'patient': patient_id,
        'exercise': active_exercise,
        'n_reps': len(results),
        'n_correct': n_correct,
        'pct_correct': round(n_correct / max(len(results), 1) * 100, 1),
        'avg_quality': round(avg_q, 3),
        'avg_torque': round(avg_t, 3),
        'hw_mode': HARDWARE_MODES[hardware_mode]['name'],
        'reps': results,
    }

    print(f'\nSession complete: {len(results)} reps  '
          f'correct={n_correct}/{len(results)}  '
          f'avg_quality={avg_q:.3f}')

    return summary


print('All functions loaded. ✓')
print(f'RehabNet._N_EXERCISES={RehabNet._N_EXERCISES}  '
      f'N_EXERCISES={N_EXERCISES}  '
      f'{"✓ MATCH" if RehabNet._N_EXERCISES == N_EXERCISES else "✗ MISMATCH — fix this!"}')

All functions loaded. ✓
RehabNet._N_EXERCISES=10  N_EXERCISES=10  ✓ MATCH


In [ ]:
OLD_EXERCISE_LIST = [
    'deep_squat', 'hurdle_step', 'inline_lunge', 'side_lunge',
    'sit_to_stand', 'straight_leg_raise',
    'squat', 'ctk_squat', 'unknown'
]

def adapt_exercise_weights(state, old_list, new_list):
    old_n = len(old_list)
    new_n = len(new_list)

    if old_n == new_n:
        return state

    print(f"Adapting exercise heads: {old_n} -> {new_n}")

    # ex_emb.weight
    if 'ex_emb.weight' in state:
        old_w = state['ex_emb.weight']
        new_w = torch.randn(new_n, old_w.shape[1]) * 0.02

        for old_i, name in enumerate(old_list):
            if name in new_list:
                new_i = new_list.index(name)
                new_w[new_i] = old_w[old_i]

        state['ex_emb.weight'] = new_w

    # ex_head.weight
    if 'ex_head.weight' in state:
        old_w = state['ex_head.weight']
        new_w = torch.randn(new_n, old_w.shape[1]) * 0.02

        for old_i, name in enumerate(old_list):
            if name in new_list:
                new_i = new_list.index(name)
                new_w[new_i] = old_w[old_i]

        state['ex_head.weight'] = new_w

    # ex_head.bias
    if 'ex_head.bias' in state:
        old_b = state['ex_head.bias']
        new_b = torch.zeros(new_n)

        for old_i, name in enumerate(old_list):
            if name in new_list:
                new_i = new_list.index(name)
                new_b[new_i] = old_b[old_i]

        state['ex_head.bias'] = new_b

    return state


model = RehabNet().to(DEVICE)

state = torch.load(
    '/content/drive/MyDrive/models/rehabnet_best.pth',
    map_location=DEVICE
)

# In case checkpoint was saved as {'model_state_dict': ...}
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']

state = adapt_exercise_weights(state, OLD_EXERCISE_LIST, EXERCISE_LIST)

model.load_state_dict(state, strict=True)
model.eval()

print("Model loaded successfully.")

Adapting exercise heads: 9 -> 10
Model loaded successfully.


In [ ]:
!pip install pyserial -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.6 MB/s eta 0:00:00


In [ ]:
try:
    import serial
    SERIAL_AVAILABLE = True
except ImportError:
    SERIAL_AVAILABLE = False
    print('pyserial not installed — running in stub mode')

import json

HW_PORT    = None   # set to '/dev/ttyUSB0' or 'COM3' when hardware ready
HW_BAUD    = 115200
HW_ENABLED = False  # stays False until hw_connect() succeeds
_hw_serial = None

def hw_connect(port=HW_PORT, baud=HW_BAUD):
    global _hw_serial, HW_ENABLED
    if port is None or not SERIAL_AVAILABLE:
        print(f'Hardware: stub mode — no commands will be sent')
        print(f'  (Set HW_PORT when hardware is connected)')
        return
    try:
        _hw_serial = serial.Serial(port, baud, timeout=1)
        HW_ENABLED = True
        print(f'Hardware connected: {port} @ {baud}')
    except Exception as e:
        print(f'Hardware connect failed: {e}')
        print(f'Running in stub mode')

def hw_send(hw_payload: dict, rep_id: str = ''):
    cmd = {
        'rep':             rep_id,
        'torque':         hw_payload['torque'],
        'mode':           hw_payload['mode_int'],
        'l_peak_deg':     hw_payload['joint']['left_knee_peak_deg'],
        'r_peak_deg':     hw_payload['joint']['right_knee_peak_deg'],
        'target_deg':     hw_payload['target']['knee_target_deg'],
        'l_deficit':      hw_payload['target']['left_deficit_deg'],
        'r_deficit':      hw_payload['target']['right_deficit_deg'],
        'torque_per_deg': hw_payload['torque_per_degree'],
    }
    msg = json.dumps(cmd) + '\n'
    if HW_ENABLED and _hw_serial and _hw_serial.is_open:
        try:
            _hw_serial.write(msg.encode('utf-8'))
            return
        except Exception as e:
            print(f'  HW send error: {e}')
    # Stub output
    print(f'  [HW STUB] torque={cmd["torque"]:.3f}  '
          f'mode={cmd["mode"]}  target={cmd["target_deg"]}°  '
          f'L_deficit={cmd["l_deficit"]}°  R_deficit={cmd["r_deficit"]}°')

def hw_disconnect():
    global _hw_serial, HW_ENABLED
    if _hw_serial and _hw_serial.is_open:
        _hw_serial.close()
        HW_ENABLED = False
        print('Hardware disconnected')

hw_connect()   # prints stub mode message — that's fine for now

Hardware: stub mode — no commands will be sent
  (Set HW_PORT when hardware is connected)


In [ ]:
EX_MIN_DIST.update({
    'inline_lunge': 100,
    'side_lunge': 100,
    'deep_squat': 60,
    'sit_to_stand': 80,
    'straight_leg_raise': 40,
    'knee_bend': 60,
    'squat': 60,
    'ctk_squat': 60,
    'hurdle_step': 45,
    'unknown': 60,
    'default': 60,
})

EX_MIN_ROM = {
    'inline_lunge': 35.0,
    'side_lunge': 35.0,
    'deep_squat': 30.0,
    'sit_to_stand': 35.0,
    'straight_leg_raise': 25.0,
    'knee_bend': 30.0,
    'squat': 30.0,
    'ctk_squat': 30.0,
    'hurdle_step': 20.0,
    'unknown': 30.0,
    'default': 30.0,
}

def _segment_reps(df, angle_col='angle_knee_L_smooth',
                  min_frames=MIN_FRAMES, min_rom=None, exercise='unknown'):
    if angle_col in df.columns:
        angles_l = df[angle_col].ffill().bfill().values
    else:
        angles_l = _knee_angle_from_df(df, 'L')

    try:
        angles_r = _knee_angle_from_df(df, 'R')
    except Exception:
        angles_r = angles_l

    angles_l = np.asarray(angles_l, dtype=np.float32)
    angles_r = np.asarray(angles_r, dtype=np.float32)

    n = len(angles_l)
    if n < min_frames:
        print(f'  Segmenter: too short n_frames={n}')
        return []

    ex_key = exercise if exercise in EX_MIN_DIST else 'default'
    min_dist = max(EX_MIN_DIST[ex_key], n // 20)
    min_rom = EX_MIN_ROM.get(ex_key, EX_MIN_ROM['default']) if min_rom is None else min_rom

    rom_l = float(np.nanmax(angles_l) - np.nanmin(angles_l))
    rom_r = float(np.nanmax(angles_r) - np.nanmin(angles_r))
    rom = max(rom_l, rom_r)

    # Motion gate: reject standing still / tiny pose jitter
    if rom < min_rom:
        print(f'  Segmenter: exercise={exercise}  n_frames={n}  '
              f'min_dist={min_dist}  ROM_L={rom_l:.1f}  ROM_R={rom_r:.1f}  '
              f'NO VALID MOVEMENT')
        return []

    # Use the side with larger ROM for segmentation
    angles = angles_l if rom_l >= rom_r else angles_r

    peaks, props = find_peaks(
        -angles,
        distance=min_dist,
        prominence=max(12.0, min_rom * 0.6),
        width=max(5, min_dist // 5),
    )

    print(f'  Segmenter: exercise={exercise}  n_frames={n}  '
          f'min_dist={min_dist}  ROM_L={rom_l:.1f}  ROM_R={rom_r:.1f}  '
          f'peaks={len(peaks)}')

    if len(peaks) == 0:
        print('  Segmenter: movement exists but no clean rep peaks')
        return []

    mids = [(peaks[i] + peaks[i + 1]) // 2 for i in range(len(peaks) - 1)]
    bounds = [0] + mids + [n]

    reps = []
    for i in range(len(bounds) - 1):
        rep = df.iloc[bounds[i]:bounds[i + 1]].copy()
        if len(rep) < min_frames:
            continue

        try:
            rep_l = _knee_angle_from_df(rep, 'L')
            rep_r = _knee_angle_from_df(rep, 'R')
            rep_rom = max(
                float(np.nanmax(rep_l) - np.nanmin(rep_l)),
                float(np.nanmax(rep_r) - np.nanmin(rep_r)),
            )
        except Exception:
            rep_rom = rom

        if rep_rom >= min_rom:
            reps.append(rep)
        else:
            print(f'    reject rep {i+1}: ROM={rep_rom:.1f} < {min_rom:.1f}')

    return reps

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

CLIP_SECONDS  = 10
LIVE_CLIP     = '/content/live_clip.webm'
PATIENT_ID    = 'LIVE_001'
EXERCISE      = 'squat'
HARDWARE_MODE = 0

def record_video(filename=LIVE_CLIP, seconds=CLIP_SECONDS):
    js = Javascript(f'''
    async function recordVideo() {{
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      video.width = 480;
      video.muted = true;
      video.autoplay = true;
      div.appendChild(video);
      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({{video: true, audio: false}});
      video.srcObject = stream;
      await video.play();

      const recorder = new MediaRecorder(stream, {{mimeType: 'video/webm'}});
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);

      const statusDiv = document.createElement('div');
      statusDiv.style.fontSize = '20px';
      statusDiv.style.fontWeight = 'bold';
      div.appendChild(statusDiv);

      recorder.start();

      for (let t = {seconds}; t > 0; t--) {{
        statusDiv.innerText = `Recording... ${{t}}s remaining`;
        await new Promise(r => setTimeout(r, 1000));
      }}

      recorder.stop();
      await new Promise(r => recorder.onstop = r);

      stream.getTracks().forEach(t => t.stop());
      div.remove();

      const blob = new Blob(chunks, {{type: 'video/webm'}});
      const buffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(buffer);
      let binary = '';
      for (let i = 0; i < bytes.byteLength; i++) {{
        binary += String.fromCharCode(bytes[i]);
      }}
      return btoa(binary);
    }}
    ''')
    display(js)
    print(f"Allow camera access in the popup. Recording for {seconds}s...")
    data = eval_js('recordVideo()')
    binary = b64decode(data)
    with open(filename, 'wb') as f:
        f.write(binary)
    print(f"Saved: {filename}  ({len(binary)/1024:.1f} KB)")
    return filename

clip_path = record_video(LIVE_CLIP, CLIP_SECONDS)

summary = run_patient_video(
    video_path    = clip_path,
    patient_id    = PATIENT_ID,
    model         = model,
    exercise      = EXERCISE,
    hardware_mode = HARDWARE_MODE,
    device        = DEVICE,
    debug         = True,
)

for rep in summary.get('reps', []):
    hw_send(rep['hardware'], rep_id=rep['rep_id'])
    print(f"  {rep['rep_id']:20s} {'✓' if rep['correct'] else '✗'}"
          f"  q={rep['quality_score']:.2f}  \"{rep['feedback']}\"")

<IPython.core.display.Javascript object>

Allow camera access in the popup. Recording for 10s...
Saved: /content/live_clip.webm  (3851.5 KB)

Patient: LIVE_001

[PS1] Processing video...
  Downloaded.


KeyboardInterrupt: 

In [ ]:
results = run_patient_video(
    video_path=clip_path,
    patient_id=PATIENT_ID,
    model=model,
    device=DEVICE,
    exercise=EXERCISE,
    hardware_mode=HARDWARE_MODE,
    debug=True
)
print(results)

TypeError: run_patient_video() got an unexpected keyword argument 'video_path'

In [ ]:
# ── NEW WRAPPER: bridges video file → new run_patient_video() ──────────────
def run_patient_video_from_path(video_path, patient_id, model, device,
                                exercise=None, fps=25.0, debug=True):
    """
    Drop-in replacement that takes a video path (like the old pipeline)
    but uses the new corrected segmentation/inference internals.
    """
    print(f"\n{'='*60}\nPatient: {patient_id}\n{'='*60}")

    # ── Step 1: PS1 — video → df with joint angles ──────────────────────
    print("\n[PS1] Processing video...")
    df = ps1_process_video(video_path, patient_id, exercise or 'unknown')

    # ── Step 2: Build kps_array aligned with df rows ────────────────────
    # df_to_kps converts the 6-joint xyz columns into (N,6,3)
    kps_array = _df_to_kps(df)   # (N_frames, 6, 3)

    # ── Step 3: rename angle columns to match new pipeline expectations ─
    # New pipeline expects 'l_knee_angle' / 'r_knee_angle'
    # Old PS1 produces 'angle_knee_L_smooth' / 'angle_knee_R_smooth'
    df = df.copy()
    df['l_knee_angle'] = df['angle_knee_L_smooth']
    df['r_knee_angle'] = df['angle_knee_R_smooth']

    # ── Step 4: call the NEW run_patient_video ──────────────────────────
    results = run_patient_video(
        df_full   = df,
        model     = model,
        device    = device,
        exercise  = exercise,
        fps       = fps,
        kps_array = kps_array,
        debug     = debug,
    )

    return results

In [ ]:
results = run_patient_video_from_path(
    video_path = clip_path,
    patient_id = PATIENT_ID,
    model      = model,
    device     = DEVICE,
    exercise   = None,   # auto-classify
)


Patient: LIVE_001

[PS1] Processing video...
  PS1: 369 frames  detection=100%  29.6s
[Pipeline] No exercise specified — running rough segmentation to classify.
[Segmenter] exercise=unknown  n_frames=369  min_dist=60  ROM_L=100.4°  ROM_R=24.9°
[Segmenter] peaks=2 at frames [52, 271]  prominence_threshold=6.0°
[Segmenter] 2 valid reps extracted.
[Scalar Debug] L_ROM=50.6°  R_ROM=15.0°  L_peak=109.3°  R_peak=76.7°  Symmetry=70.3  Vel=12.3°/s  Jerk=2603.88  Lag=58fr  Smooth=0.000
[Pipeline] Auto-classified exercise: straight_leg_raise
[Segmenter] exercise=straight_leg_raise  n_frames=369  min_dist=40  ROM_L=100.4°  ROM_R=24.9°
[Segmenter] peaks=2 at frames [271, 315]  prominence_threshold=6.0°
[Segmenter] 2 valid reps extracted.
[Pipeline] 2 reps to process.

[Pipeline] ── Rep 1/2 ──
[Scalar Debug] L_ROM=96.0°  R_ROM=19.7°  L_peak=63.9°  R_peak=74.2°  Symmetry=79.5  Vel=22.4°/s  Jerk=5267.04  Lag=1fr  Smooth=0.000
[Pipeline] Rep 1: INCORRECT  quality=0.31  hw_mode=assist  feedback=['Bend

In [ ]:
def safe_infer(model, kps_np, scalars, exercise, device, exercise_to_idx=None):
    import torch
    if exercise_to_idx is None:
        exercise_to_idx = EXERCISE_TO_IDX

    ex_idx = exercise_to_idx.get(exercise, exercise_to_idx.get('unknown', 0))

    kps_t = torch.tensor(kps_to_model_input(kps_np), device=device)  # (1,3,150,6)
    sc_t  = torch.tensor(scalars[np.newaxis], device=device)          # (1,10)
    ex_t  = torch.tensor([ex_idx], dtype=torch.long, device=device)
    adj_t = torch.from_numpy(ADJ).to(device)                          # ← add this
    mode_t = torch.zeros(1, dtype=torch.long, device=device)          # ← add this

    if torch.isnan(kps_t).any() or torch.isnan(sc_t).any():
        print("[safe_infer] NaN detected — skipping.")
        return None

    model.eval()
    with torch.no_grad():
        try:
            cls_logits, quality_out, ex_logits = model(kps_t, adj_t, sc_t, mode_t, ex_t)

            pred_label    = int(cls_logits.argmax(dim=1).item())
            quality_score = float(torch.sigmoid(quality_out).squeeze().item())
            ex_pred_idx   = int(ex_logits.argmax(dim=1).item())
            ex_pred_name  = EXERCISE_LIST[ex_pred_idx] if ex_pred_idx < len(EXERCISE_LIST) else 'unknown'

            return {
                'correct':       pred_label == 1,   # ← 1=correct in your RehabNet, not 0
                'pred_label':    pred_label,
                'quality_score': quality_score,
                'exercise_pred': ex_pred_name,
                'cls_logits':    cls_logits.cpu().numpy(),
            }
        except Exception as e:
            print(f"[safe_infer] Model error: {e}")
            return None

In [ ]:
from scipy.signal import savgol_filter

def _smooth_signal(signal, win=9, poly=2):
    if len(signal) < win:
        win = len(signal) - 1 if len(signal) % 2 == 0 else len(signal)
        if win < poly + 2:
            return signal
    if win % 2 == 0:
        win += 1
    return savgol_filter(signal, win, poly)

def _velocity(signal, fps=25.0):
    smoothed = _smooth_signal(signal)
    return np.abs(np.gradient(smoothed, 1.0/fps))

def _jerk(signal, fps=25.0):
    smoothed = _smooth_signal(signal)
    vel = np.gradient(smoothed, 1.0/fps)
    acc = _smooth_signal(vel)
    acc = np.gradient(acc, 1.0/fps)
    return np.abs(np.gradient(acc, 1.0/fps))

In [ ]:
# ── CELL: Upload a fresh video and confirm it's new ─────────────────────────
from google.colab import files
import os, time

print("Select your video file...")
uploaded = files.upload()

# Get the uploaded filename
clip_path = list(uploaded.keys())[0]
clip_path = os.path.abspath(clip_path)

print(f"\nclip_path = {clip_path}")
print(f"exists:    {os.path.exists(clip_path)}")
print(f"size:      {os.path.getsize(clip_path)/1e6:.2f} MB")
print(f"modified:  {time.ctime(os.path.getmtime(clip_path))}")
print(f"NOW:       {time.ctime(time.time())}")

Select your video file...


Saving 8837221-uhd_2160_4096_25fps.mp4 to 8837221-uhd_2160_4096_25fps (1).mp4

clip_path = /content/8837221-uhd_2160_4096_25fps (1).mp4
exists:    True
size:      16.18 MB
modified:  Fri Jun 12 18:13:29 2026
NOW:       Fri Jun 12 18:13:29 2026


In [ ]:
# ── DROP-IN FIXES: smoothed derivatives + unilateral exercise guard ─────────
from scipy.signal import savgol_filter
import numpy as np

UNILATERAL_EXERCISES = {'straight_leg_raise', 'inline_lunge', 'side_lunge'}


def _smooth_signal(signal, win=9, poly=2):
    """Savitzky-Golay smoothing — required before any differentiation."""
    signal = np.asarray(signal, dtype=np.float64)
    n = len(signal)
    if n < poly + 2:
        return signal
    w = min(win, n if n % 2 == 1 else n - 1)
    if w % 2 == 0:
        w -= 1
    if w < poly + 2:
        w = poly + 2 if (poly + 2) % 2 == 1 else poly + 3
        if w > n:
            return signal
    return savgol_filter(signal, w, poly)


def _velocity(signal, fps=25.0):
    """Velocity from SMOOTHED signal — avoids noise amplification."""
    smoothed = _smooth_signal(signal)
    return np.abs(np.gradient(smoothed, 1.0 / fps))


def _jerk(signal, fps=25.0):
    """Jerk from SMOOTHED velocity/acceleration — avoids 3x noise amplification."""
    smoothed = _smooth_signal(signal)
    vel = np.gradient(smoothed, 1.0 / fps)
    vel_s = _smooth_signal(vel)
    acc = np.gradient(vel_s, 1.0 / fps)
    acc_s = _smooth_signal(acc)
    return np.abs(np.gradient(acc_s, 1.0 / fps))


def generate_feedback(scalars: np.ndarray, exercise: str = 'squat') -> list:
    """
    Rule-based feedback — now skips symmetry/lag checks for
    unilateral exercises (SLR, lunges) where one leg is
    intentionally stationary.
    """
    l_rom_deg  = scalars[0] * 180.0
    r_rom_deg  = scalars[1] * 180.0
    l_peak_deg = scalars[2] * 180.0
    r_peak_deg = scalars[3] * 180.0
    symmetry   = scalars[4] * 100.0
    mean_vel   = scalars[5] * 200.0
    mean_jerk  = scalars[6] * 10.0
    temp_lag   = scalars[8] * 150.0
    smoothness = scalars[9]

    feedback = []

    ROM_TARGETS = {
        'squat':              90.0,
        'deep_squat':        100.0,
        'inline_lunge':       80.0,
        'side_lunge':         70.0,
        'sit_to_stand':       90.0,
        'straight_leg_raise': 60.0,
        'knee_bend':          90.0,
        'ctk_squat':          90.0,
        'hurdle_step':        70.0,
        'default':            70.0,
    }
    target_rom = ROM_TARGETS.get(exercise, ROM_TARGETS['default'])

    # ── For unilateral exercises, only check the ACTIVE leg ─────────────
    if exercise in UNILATERAL_EXERCISES:
        active_rom = max(l_rom_deg, r_rom_deg)
        if active_rom < target_rom * 0.75:
            feedback.append("Bend your knee further — aim for a deeper range of motion.")
    else:
        if l_rom_deg < target_rom * 0.75 or r_rom_deg < target_rom * 0.75:
            feedback.append("Bend your knee further — aim for a deeper range of motion.")

    # ── too fast ─────────────────────────────────────────────────────────
    if mean_vel > 120.0:
        feedback.append("Slow down — controlled movement is safer and more effective.")

    # ── high jerk ────────────────────────────────────────────────────────
    if mean_jerk > 5.0:
        feedback.append("Try to move more smoothly — avoid jerky or sudden changes.")

    # ── asymmetry / lag — SKIP for unilateral exercises ────────────────
    if exercise not in UNILATERAL_EXERCISES:
        if symmetry > 20.0:
            side = "left" if l_rom_deg < r_rom_deg else "right"
            feedback.append(f"Your {side} side is lagging — try to move both sides equally.")
        if temp_lag > 15.0:
            feedback.append("Your left and right sides are out of sync — try to move together.")

    # ── low smoothness ───────────────────────────────────────────────────
    if smoothness < 0.4:
        feedback.append("Work on a smooth, continuous movement pattern.")

    if not feedback:
        feedback.append("Good form — keep it up!")

    return feedback


print("Drop-in fixes applied: smoothing + unilateral exercise guard")

Drop-in fixes applied: smoothing + unilateral exercise guard


In [ ]:
# ── CELL: Run pipeline on freshly uploaded video ─────────────────────────────
results = run_patient_video_from_path(
    video_path = clip_path,
    patient_id = PATIENT_ID,
    model      = model,
    device     = DEVICE,
    exercise   = None,   # auto-classify
)


Patient: LIVE_001

[PS1] Processing video...
  PS1: 639 frames  detection=100%  61.0s
[Pipeline] No exercise specified — running rough segmentation to classify.
[Segmenter] exercise=unknown  n_frames=639  min_dist=60  ROM_L=123.3°  ROM_R=103.1°
[Segmenter] peaks=5 at frames [50, 168, 304, 437, 569]  prominence_threshold=6.0°
[Segmenter] 5 valid reps extracted.
[Scalar Debug] L_ROM=100.2°  R_ROM=81.3°  L_peak=60.7°  R_peak=74.7°  Symmetry=18.9  Vel=32.3°/s  Jerk=1795.01  Lag=14fr  Smooth=0.205
[Pipeline] Auto-classified exercise: inline_lunge
[Segmenter] exercise=inline_lunge  n_frames=639  min_dist=100  ROM_L=123.3°  ROM_R=103.1°
[Segmenter] peaks=5 at frames [50, 168, 304, 437, 569]  prominence_threshold=8.0°
[Segmenter] 5 valid reps extracted.
[Pipeline] 5 reps to process.

[Pipeline] ── Rep 1/5 ──
[Scalar Debug] L_ROM=100.2°  R_ROM=81.3°  L_peak=60.7°  R_peak=74.7°  Symmetry=18.9  Vel=32.3°/s  Jerk=1795.01  Lag=14fr  Smooth=0.205
[Pipeline] Rep 1: INCORRECT  quality=0.57  hw_mode=f

In [ ]:
"""
physio_pipeline_v2.py
======================
Full corrected pipeline.
- Smoothed velocity/jerk (no noise blowup)
- Acceleration-magnitude jerk proxy (stable)
- Rule-based correct/incorrect (replaces miscalibrated binary head)
- Hook for EXTERNAL exercise classifier (replaces miscalibrated exercise head)
- Unilateral-exercise-aware feedback
"""

import numpy as np
import pandas as pd
import torch
from scipy.signal import savgol_filter, find_peaks


# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────

L_HIP, L_KNEE, L_ANK = 0, 1, 2
R_HIP, R_KNEE, R_ANK = 3, 4, 5

EXERCISE_LIST = [
    'deep_squat', 'hurdle_step', 'inline_lunge', 'side_lunge',
    'sit_to_stand', 'straight_leg_raise', 'squat', 'ctk_squat',
    'knee_bend', 'unknown',
]
EXERCISE_TO_IDX = {ex: i for i, ex in enumerate(EXERCISE_LIST)}

UNILATERAL_EXERCISES = {'straight_leg_raise', 'inline_lunge', 'side_lunge'}

EX_MIN_DIST = {
    'inline_lunge': 100, 'side_lunge': 100, 'deep_squat': 60,
    'sit_to_stand': 80, 'straight_leg_raise': 40, 'knee_bend': 60,
    'squat': 60, 'ctk_squat': 60, 'hurdle_step': 45,
    'unknown': 60, 'default': 60,
}

EX_MIN_ROM = {
    'inline_lunge': 20.0, 'side_lunge': 20.0, 'deep_squat': 25.0,
    'sit_to_stand': 20.0, 'straight_leg_raise': 15.0, 'knee_bend': 15.0,
    'squat': 20.0, 'ctk_squat': 20.0, 'hurdle_step': 15.0,
    'unknown': 15.0, 'default': 15.0,
}

ROM_TARGETS = {
    'squat': 80.0, 'deep_squat': 80.0, 'inline_lunge': 70.0,
    'side_lunge': 60.0, 'sit_to_stand': 70.0,
    'straight_leg_raise': 45.0, 'knee_bend': 80.0,
    'ctk_squat': 80.0, 'hurdle_step': 70.0, 'default': 60.0,
}


# ─────────────────────────────────────────────
# GEOMETRY HELPERS
# ─────────────────────────────────────────────

def _safe_angle_deg(a, b, c):
    v1, v2 = a - b, c - b
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-8 or n2 < 1e-8:
        return 0.0
    cos_a = np.dot(v1, v2) / (n1 * n2)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def _compute_knee_angles(kps_np):
    T = kps_np.shape[0]
    angles_L = np.zeros(T)
    angles_R = np.zeros(T)
    for t in range(T):
        f = kps_np[t]
        angles_L[t] = _safe_angle_deg(f[L_HIP], f[L_KNEE], f[L_ANK])
        angles_R[t] = _safe_angle_deg(f[R_HIP], f[R_KNEE], f[R_ANK])
    return angles_L, angles_R


# ─────────────────────────────────────────────
# SMOOTHED DERIVATIVES (the actual fix)
# ─────────────────────────────────────────────

def _smooth_signal(signal, win=15, poly=3):
    signal = np.asarray(signal, dtype=np.float64)
    n = len(signal)
    if n < poly + 2:
        return signal
    w = min(win, n if n % 2 == 1 else n - 1)
    if w % 2 == 0:
        w -= 1
    if w < poly + 2:
        w = poly + 2 if (poly + 2) % 2 == 1 else poly + 3
        if w > n:
            return signal
    return savgol_filter(signal, w, poly)


def _velocity(signal, fps=25.0):
    smoothed = _smooth_signal(signal, win=15, poly=3)
    return np.abs(np.gradient(smoothed, 1.0 / fps))


def _jerk(signal, fps=25.0):
    """
    Acceleration-magnitude proxy for jerk.
    Triple-derivative is too unstable on 150-frame MediaPipe data.
    This is smoothed velocity → smoothed acceleration (2 derivatives, not 3).
    Typical range: 0-30. Normalize by /30.0 in extract_scalars.
    """
    smoothed = _smooth_signal(signal, win=15, poly=3)
    vel      = np.gradient(smoothed, 1.0 / fps)
    vel_s    = _smooth_signal(vel, win=15, poly=3)
    acc      = np.gradient(vel_s, 1.0 / fps)
    return np.abs(acc)


# ─────────────────────────────────────────────
# extract_scalars
# ─────────────────────────────────────────────

def extract_scalars(kps_np, exercise='squat', fps=25.0, debug=True):
    kps_np = np.array(kps_np, dtype=np.float64)
    angles_L, angles_R = _compute_knee_angles(kps_np)

    l_rom_deg  = float(angles_L.max() - angles_L.min())
    r_rom_deg  = float(angles_R.max() - angles_R.min())
    l_peak_deg = float(angles_L.min())
    r_peak_deg = float(angles_R.min())

    max_rom  = max(l_rom_deg, r_rom_deg, 1e-8)
    symmetry = abs(l_rom_deg - r_rom_deg) / max_rom * 100.0

    mean_angle = (angles_L + angles_R) / 2.0
    vel        = _velocity(mean_angle, fps)
    jrk        = _jerk(mean_angle, fps)
    mean_vel   = float(vel.mean())
    mean_jerk  = float(jrk.mean())

    T         = len(angles_L)
    l_min_idx = int(np.argmin(angles_L))
    r_min_idx = int(np.argmin(angles_R))
    temp_lag  = float(abs(l_min_idx - r_min_idx))

    smoothness = float(1.0 - np.std(vel) / (mean_vel + 1e-8))
    smoothness = float(np.clip(smoothness, 0.0, 1.0))

    trunk_lean = 0.0

    if debug:
        print(
            f"[Scalar Debug] "
            f"L_ROM={l_rom_deg:.1f}°  R_ROM={r_rom_deg:.1f}°  "
            f"L_peak={l_peak_deg:.1f}°  R_peak={r_peak_deg:.1f}°  "
            f"Symmetry={symmetry:.1f}  Vel={mean_vel:.1f}°/s  "
            f"Jerk={mean_jerk:.2f}  Lag={temp_lag:.0f}fr  "
            f"Smooth={smoothness:.3f}"
        )

    scalars = np.array([
        l_rom_deg  / 180.0,
        r_rom_deg  / 180.0,
        l_peak_deg / 180.0,
        r_peak_deg / 180.0,
        symmetry   / 100.0,
        mean_vel   / 200.0,
        mean_jerk  / 30.0,     # ← changed from /10.0
        trunk_lean,
        temp_lag   / 150.0,
        smoothness,
    ], dtype=np.float32)

    return np.clip(scalars, 0.0, 2.0)


# ─────────────────────────────────────────────
# segment_reps
# ─────────────────────────────────────────────

def segment_reps(df, exercise='squat', fps=25.0, debug=True):
    ex_key   = exercise if exercise in EX_MIN_DIST else 'default'
    min_dist = EX_MIN_DIST[ex_key]
    min_rom  = EX_MIN_ROM[ex_key]

    angles_L = df['l_knee_angle'].values.astype(np.float64)
    angles_R = df['r_knee_angle'].values.astype(np.float64)
    n_frames = len(angles_L)

    rom_L = float(angles_L.max() - angles_L.min())
    rom_R = float(angles_R.max() - angles_R.min())

    if debug:
        print(f"[Segmenter] exercise={exercise}  n_frames={n_frames}  "
              f"min_dist={min_dist}  ROM_L={rom_L:.1f}°  ROM_R={rom_R:.1f}°")

    if max(rom_L, rom_R) < min_rom:
        print(f"[Segmenter] No real movement (ROM_L={rom_L:.1f}°  "
              f"ROM_R={rom_R:.1f}°  threshold={min_rom}°). Returning 0 reps.")
        return []

    mean_angle = (angles_L + angles_R) / 2.0
    signal_inv = -mean_angle
    prominence = max(min_rom * 0.4, 5.0)

    peaks, _ = find_peaks(signal_inv, distance=min_dist, prominence=prominence)

    if debug:
        print(f"[Segmenter] peaks={len(peaks)} at frames {peaks.tolist()}  "
              f"prominence_threshold={prominence:.1f}°")

    if len(peaks) == 0:
        print("[Segmenter] Movement found but no clear peaks. Treating as 1 rep.")
        return [df.copy()]

    boundaries = [0]
    for i in range(len(peaks) - 1):
        boundaries.append((peaks[i] + peaks[i + 1]) // 2)
    boundaries.append(n_frames)

    reps = []
    for i in range(len(peaks)):
        start, end = boundaries[i], boundaries[i + 1]
        rep_df = df.iloc[start:end].copy()
        r_rom_L = rep_df['l_knee_angle'].max() - rep_df['l_knee_angle'].min()
        r_rom_R = rep_df['r_knee_angle'].max() - rep_df['r_knee_angle'].min()
        if max(r_rom_L, r_rom_R) < min_rom:
            if debug:
                print(f"[Segmenter] Rep {i+1} skipped: ROM too small "
                      f"({max(r_rom_L, r_rom_R):.1f}°)")
            continue
        reps.append(rep_df)

    if debug:
        print(f"[Segmenter] {len(reps)} valid reps extracted.")
    return reps


# ─────────────────────────────────────────────
# RULE-BASED LABEL (replaces miscalibrated binary head)
# ─────────────────────────────────────────────

def rule_based_label(scalars, exercise='inline_lunge'):
    l_rom = scalars[0] * 180.0
    r_rom = scalars[1] * 180.0
    sym   = scalars[4] * 100.0
    vel   = scalars[5] * 200.0
    jerk  = scalars[6] * 30.0

    target     = ROM_TARGETS.get(exercise, ROM_TARGETS['default'])
    active_rom = max(l_rom, r_rom)

    reasons = []
    if active_rom < target * 0.7:
        reasons.append('insufficient_rom')
    if sym > 35.0 and exercise not in UNILATERAL_EXERCISES:
        reasons.append('asymmetric')
    if jerk > 15.0:
        reasons.append('jerky')
    if vel > 150.0:
        reasons.append('too_fast')

    return (1 if not reasons else 0), reasons


# ─────────────────────────────────────────────
# generate_feedback
# ─────────────────────────────────────────────

def generate_feedback(scalars, exercise='squat'):
    l_rom_deg  = scalars[0] * 180.0
    r_rom_deg  = scalars[1] * 180.0
    symmetry   = scalars[4] * 100.0
    mean_vel   = scalars[5] * 200.0
    mean_jerk  = scalars[6] * 30.0
    temp_lag   = scalars[8] * 150.0
    smoothness = scalars[9]

    feedback   = []
    target_rom = ROM_TARGETS.get(exercise, ROM_TARGETS['default'])

    if exercise in UNILATERAL_EXERCISES:
        active_rom = max(l_rom_deg, r_rom_deg)
        if active_rom < target_rom * 0.75:
            feedback.append("Bend your knee further — aim for a deeper range of motion.")
    else:
        if l_rom_deg < target_rom * 0.75 or r_rom_deg < target_rom * 0.75:
            feedback.append("Bend your knee further — aim for a deeper range of motion.")

    if mean_vel > 120.0:
        feedback.append("Slow down — controlled movement is safer and more effective.")

    if mean_jerk > 15.0:
        feedback.append("Try to move more smoothly — avoid jerky or sudden changes.")

    if exercise not in UNILATERAL_EXERCISES:
        if symmetry > 20.0:
            side = "left" if l_rom_deg < r_rom_deg else "right"
            feedback.append(f"Your {side} side is lagging — try to move both sides equally.")
        if temp_lag > 15.0:
            feedback.append("Your left and right sides are out of sync — try to move together.")

    if smoothness < 0.4:
        feedback.append("Work on a smooth, continuous movement pattern.")

    if not feedback:
        feedback.append("Good form — keep it up!")

    return feedback


# ─────────────────────────────────────────────
# build_hardware_payload
# ─────────────────────────────────────────────

def build_hardware_payload(kps_np, pred_label, quality_score, exercise, feedback, fps=25.0):
    angles_L, angles_R = _compute_knee_angles(np.array(kps_np, dtype=np.float64))

    l_rom_deg  = float(angles_L.max() - angles_L.min())
    r_rom_deg  = float(angles_R.max() - angles_R.min())
    l_peak_deg = float(angles_L.min())
    r_peak_deg = float(angles_R.min())

    mean_angle = (angles_L + angles_R) / 2.0
    mean_vel   = float(_velocity(mean_angle, fps).mean())

    is_correct = (pred_label == 1)
    if is_correct and quality_score >= 0.7:
        hw_mode, torque = 'resist', round(quality_score * 0.5, 3)
    elif not is_correct or quality_score < 0.4:
        hw_mode, torque = 'assist', round((1.0 - quality_score) * 0.3, 3)
    else:
        hw_mode, torque = 'free', 0.0

    return {
        'exercise':           exercise,
        'correct':            is_correct,
        'quality_score':      round(float(quality_score), 4),
        'left_knee_rom_deg':  round(l_rom_deg, 2),
        'right_knee_rom_deg': round(r_rom_deg, 2),
        'left_knee_peak_deg': round(l_peak_deg, 2),
        'right_knee_peak_deg':round(r_peak_deg, 2),
        'mean_velocity_dps':  round(mean_vel, 2),
        'hw_mode':            hw_mode,
        'torque_signal':      torque,
        'feedback':           feedback,
    }


# ─────────────────────────────────────────────
# resample / shape helpers
# ─────────────────────────────────────────────

def resample_rep(kps_np, target_frames=150):
    T = kps_np.shape[0]
    if T == target_frames:
        return kps_np.copy()
    old_idx = np.linspace(0, T - 1, T)
    new_idx = np.linspace(0, T - 1, target_frames)
    out = np.zeros((target_frames, 6, 3), dtype=np.float32)
    for j in range(6):
        for c in range(3):
            out[:, j, c] = np.interp(new_idx, old_idx, kps_np[:, j, c])
    return out


def kps_to_model_input(kps_np):
    kps = kps_np.transpose(2, 0, 1).astype(np.float32)
    return kps[np.newaxis, ...]


# ─────────────────────────────────────────────
# EXTERNAL EXERCISE CLASSIFIER HOOK
# ─────────────────────────────────────────────

def predict_exercise_external(kps_np, scalars, classifier_model=None, classifier_predict_fn=None):
    """
    Replace the model's internal exercise_classify_head with your
    SEPARATE exercise classifier here.

    Provide EITHER:
      classifier_model     : your loaded model object
      classifier_predict_fn: callable(kps_np, scalars) -> exercise_name (str)

    Returns: exercise_name (str)
    """
    if classifier_predict_fn is not None:
        return classifier_predict_fn(kps_np, scalars)

    if classifier_model is not None:
        # TODO: replace with actual call signature once you share the model
        # e.g. pred = classifier_model.predict(features)
        raise NotImplementedError(
            "Send your classifier's code/call signature — "
            "this will be filled in to call it correctly."
        )

    # Fallback: no classifier provided
    return 'unknown'


# ─────────────────────────────────────────────
# safe_infer — quality score from RehabNet, label from rules
# ─────────────────────────────────────────────

def safe_infer(model, adj, kps_np, scalars, exercise, device, exercise_to_idx=None):
    if exercise_to_idx is None:
        exercise_to_idx = EXERCISE_TO_IDX

    ex_idx = exercise_to_idx.get(exercise, exercise_to_idx.get('unknown', 0))

    kps_t  = torch.tensor(kps_to_model_input(kps_np), device=device)
    sc_t   = torch.tensor(scalars[np.newaxis], device=device)
    ex_t   = torch.tensor([ex_idx], dtype=torch.long, device=device)
    adj_t  = torch.as_tensor(adj, device=device)
    mode_t = torch.zeros(1, dtype=torch.long, device=device)

    if torch.isnan(kps_t).any() or torch.isnan(sc_t).any():
        print("[safe_infer] NaN detected — skipping.")
        return None

    model.eval()
    with torch.no_grad():
        try:
            cls_logits, quality_out, ex_logits = model(kps_t, adj_t, sc_t, mode_t, ex_t)
            quality_score = float(torch.sigmoid(quality_out).squeeze().item())
            model_label   = int(cls_logits.argmax(dim=1).item())

            # Rule-based label (replaces miscalibrated binary head)
            rule_label, reasons = rule_based_label(scalars, exercise)

            return {
                'correct':       rule_label == 1,
                'pred_label':    rule_label,
                'reasons':       reasons,
                'model_label':   model_label,   # for debugging/comparison
                'quality_score': quality_score,
                'cls_logits':    cls_logits.cpu().numpy(),
            }
        except Exception as e:
            print(f"[safe_infer] Model error: {e}")
            return None


# ─────────────────────────────────────────────
# run_patient_video — main pipeline
# ─────────────────────────────────────────────

def run_patient_video(df_full, model, adj, device, exercise=None, fps=25.0,
                      kps_array=None, debug=True,
                      classifier_predict_fn=None):
    results = []

    # ── Step 1: exercise selection ──────────────────────────────────────
    if exercise is not None:
        print(f"[Pipeline] Exercise manually set: {exercise}")
        ex_final = exercise
    else:
        print("[Pipeline] No exercise specified — running rough segmentation to classify.")
        rough_reps = segment_reps(df_full, exercise='unknown', fps=fps, debug=debug)
        if not rough_reps:
            print("[Pipeline] No movement detected in video.")
            return []

        rough_df  = rough_reps[0]
        rough_idx = rough_df.index.tolist()
        if kps_array is not None:
            rough_kps   = kps_array[rough_idx[0]:rough_idx[-1] + 1]
            rough_kps_r = resample_rep(rough_kps)
            rough_sc    = extract_scalars(rough_kps_r, exercise='unknown', fps=fps, debug=debug)

            ex_final = predict_exercise_external(
                rough_kps_r, rough_sc, classifier_predict_fn=classifier_predict_fn
            )
            print(f"[Pipeline] Auto-classified exercise: {ex_final}")
        else:
            ex_final = 'unknown'
            print("[Pipeline] No kps_array provided — using 'unknown'.")

    # ── Step 2: re-segment with exercise-specific parameters ────────────
    reps = segment_reps(df_full, exercise=ex_final, fps=fps, debug=debug)
    if not reps:
        print("[Pipeline] No valid reps found after re-segmentation.")
        return []

    print(f"[Pipeline] {len(reps)} reps to process.")

    # ── Step 3: per-rep inference ────────────────────────────────────────
    n_correct = 0
    for i, rep_df in enumerate(reps):
        print(f"\n[Pipeline] ── Rep {i+1}/{len(reps)} ──")

        if kps_array is not None:
            idx = rep_df.index.tolist()
            rep_kps_raw = kps_array[idx[0]:idx[-1] + 1]
        else:
            print("[Pipeline] Warning: no kps_array provided — skipping.")
            rep_kps_raw = None

        if rep_kps_raw is None or len(rep_kps_raw) < 5:
            print(f"[Pipeline] Rep {i+1}: too short, skipping.")
            continue

        rep_kps = resample_rep(rep_kps_raw, target_frames=150)
        scalars = extract_scalars(rep_kps, exercise=ex_final, fps=fps, debug=debug)

        result = safe_infer(model, adj, rep_kps, scalars, ex_final, device)
        if result is None:
            print(f"[Pipeline] Rep {i+1}: inference failed, skipping.")
            continue

        feedback = generate_feedback(scalars, exercise=ex_final)
        payload  = build_hardware_payload(
            kps_np=rep_kps, pred_label=result['pred_label'],
            quality_score=result['quality_score'], exercise=ex_final,
            feedback=feedback, fps=fps,
        )

        if result['correct']:
            n_correct += 1

        print(
            f"[Pipeline] Rep {i+1}: "
            f"{'CORRECT' if result['correct'] else 'INCORRECT'}  "
            f"(reasons={result['reasons']})  "
            f"quality={result['quality_score']:.2f}  "
            f"hw_mode={payload['hw_mode']}  "
            f"feedback={feedback}"
        )

        results.append({
            'rep': i + 1, 'result': result, 'scalars': scalars,
            'payload': payload, 'feedback': feedback,
        })

    print(f"\n[Pipeline] Done. {n_correct}/{len(results)} reps correct.")
    return results


# ─────────────────────────────────────────────
# wrapper — video path → df + kps_array → run_patient_video
# ─────────────────────────────────────────────

def run_patient_video_from_path(video_path, patient_id, model, adj, device,
                                exercise=None, fps=25.0, debug=True,
                                classifier_predict_fn=None,
                                ps1_process_video_fn=None, df_to_kps_fn=None):
    print(f"\n{'='*60}\nPatient: {patient_id}\n{'='*60}")

    print("\n[PS1] Processing video...")
    df = ps1_process_video_fn(video_path, patient_id, exercise or 'unknown')

    kps_array = df_to_kps_fn(df)

    df = df.copy()
    df['l_knee_angle'] = df['angle_knee_L_smooth']
    df['r_knee_angle'] = df['angle_knee_R_smooth']

    return run_patient_video(
        df_full=df, model=model, adj=adj, device=device,
        exercise=exercise, fps=fps, kps_array=kps_array, debug=debug,
        classifier_predict_fn=classifier_predict_fn,
    )

In [ ]:
# ── CELL: Run pipeline on freshly uploaded video ─────────────────────────────
results = run_patient_video_from_path(
    video_path         = clip_path,
    patient_id         = PATIENT_ID,
    model              = model,
    adj                = ADJ, # Added the missing 'adj' argument
    device             = DEVICE,
    exercise           = None,   # auto-classify
    ps1_process_video_fn = ps1_process_video, # Pass the function here
    df_to_kps_fn       = _df_to_kps       # Pass the function here
)


Patient: LIVE_001

[PS1] Processing video...
  PS1: 639 frames  detection=100%  61.4s
[Pipeline] No exercise specified — running rough segmentation to classify.
[Segmenter] exercise=unknown  n_frames=639  min_dist=60  ROM_L=123.3°  ROM_R=103.1°
[Segmenter] peaks=5 at frames [50, 168, 304, 437, 569]  prominence_threshold=6.0°
[Segmenter] 5 valid reps extracted.
[Scalar Debug] L_ROM=100.2°  R_ROM=81.3°  L_peak=60.7°  R_peak=74.7°  Symmetry=18.9  Vel=31.7°/s  Jerk=100.78  Lag=14fr  Smooth=0.284
[Pipeline] Auto-classified exercise: unknown
[Segmenter] exercise=unknown  n_frames=639  min_dist=60  ROM_L=123.3°  ROM_R=103.1°
[Segmenter] peaks=5 at frames [50, 168, 304, 437, 569]  prominence_threshold=6.0°
[Segmenter] 5 valid reps extracted.
[Pipeline] 5 reps to process.

[Pipeline] ── Rep 1/5 ──
[Scalar Debug] L_ROM=100.2°  R_ROM=81.3°  L_peak=60.7°  R_peak=74.7°  Symmetry=18.9  Vel=31.7°/s  Jerk=100.78  Lag=14fr  Smooth=0.284
[Pipeline] Rep 1: INCORRECT  (reasons=['jerky'])  quality=0.49  h